# Семинар 4. CasADi: от основ до задач лекции

Весь семинар — за ноутбуком. Часть 1 — введение в пакет [CasADi](https://web.casadi.org) с самых основ: что такое символьная переменная, как выражение хранится деревом операций и почему производные получаются точными. Часть 2 — задачи лекции (Розенброк, квадратичная, цепь) решаются на CasADi, и каждое число сверяется с тем, что мы получили руками.

Установка: в корне репозитория один раз выполнить

```
uv sync --group casadi
```

Печатная шпаргалка по основам — раздел 11.1 конспекта [`lecture04.md`](lecture04.md).

## Часть 1. CasADi с нуля

**Шаг 1. Зачем это всё.** Весь сегодняшний день мы выписывали градиенты и гессианы руками. CasADi строит их сам — если видит не готовые числа, а саму формулу. Для этого формулу записывают *символьно*.

In [1]:
import casadi as ca
import numpy as np

print(ca.__version__)

3.8.0


**Шаг 2. Символьная переменная — имя, а не число.** У `x` нет значения, поэтому выражение не вычисляется: CasADi запоминает его как **дерево операций**. Печать показывает это дерево.

In [2]:
x = ca.SX.sym("x")     # переменная без значения
f = (1 - x) ** 2       # ничего не посчиталось
print(f)               # sq((1-x)) — дерево: вычесть из 1, возвести в квадрат

sq((1-x))


**Шаг 3. Деревья растут из обычных операций.** Арифметика, степени, `sin`, `exp` — всё перегружено и достраивает дерево.

In [3]:
g = ca.sin(x) * x ** 2 + ca.exp(-x)
print(g)

((sin(x)*sq(x))+exp((-x)))


**Шаг 4. Из дерева — функция.** `Function` получает список входов и список выходов и умеет прогонять по дереву числа. Дерево — рецепт, `Function` — повар. Входов и выходов может быть несколько.

In [4]:
F = ca.Function("F", [x], [f])
print("F(3) =", F(3))                      # (1-3)^2 = 4

y = ca.SX.sym("y")
G = ca.Function("G", [x, y], [x * y, x + y])
prod, summ = G(2, 3)
print("G(2,3) =", prod, summ)

F(3) = 4
G(2,3) = 6 5


**Шаг 5. Векторы.** Переменная может быть сразу вектором; к координатам обращаются индексом, векторы склеивают `vertcat`. Так записывают функции многих переменных.

In [5]:
v = ca.SX.sym("v", 3)                      # вектор из трёх неизвестных
h = v[0] ** 2 + 2 * v[1] ** 2 + 3 * v[2] ** 2
print(ca.Function("H3", [v], [h])([1, 1, 1]))   # 6
print(ca.vertcat(v, 0))                    # приклеили константу снизу

6
[v_0, v_1, v_2, 0]


**Шаг 6. Производные — точные.** `jacobian` проходит по дереву и по табличным правилам (производная квадрата, синуса, цепное правило) строит **новое дерево** — производную. Сравним с конечной разностью: у разности погрешность порядка $10^{-10}$, у CasADi — ноль машинных ошибок сверх арифметики. Это *алгоритмическое дифференцирование*; как оно устроено и почему почти бесплатно — лекция 7.

In [6]:
dg = ca.jacobian(g, x)                     # дерево производной
dG = ca.Function("dG", [x], [dg])
Gf = ca.Function("Gf", [x], [g])

t0, h0 = 1.3, 1e-6
exact = 2 * t0 * np.sin(t0) + t0 ** 2 * np.cos(t0) - np.exp(-t0)   # производная руками
fd = (float(Gf(t0 + h0)) - float(Gf(t0 - h0))) / (2 * h0)          # конечная разность
print(f"руками:            {exact:.15f}")
print(f"CasADi:            {float(dG(t0)):.15f}   расхождение {abs(float(dG(t0)) - exact):.1e}")
print(f"конечная разность: {fd:.15f}   расхождение {abs(fd - exact):.1e}")
assert abs(float(dG(t0)) - exact) < 1e-14

руками:            2.684792509426242
CasADi:            2.684792509426242   расхождение 0.0e+00
конечная разность: 2.684792509199951   расхождение 2.3e-10


## Часть 2. Задачи лекции на CasADi

**Задача 1. Розенброк: производные без рук.** Записываем функцию, а градиент и гессиан получаем командами `gradient` и `hessian`. В $(1,1)$ должен выйти гессиан из упражнения 12.4 — те самые $802$ и $-400$, которые на лекции считались вручную.

In [7]:
xr = ca.SX.sym("x", 2)
rosen = (1 - xr[0]) ** 2 + 100 * (xr[1] - xr[0] ** 2) ** 2

grad_rosen = ca.Function("g", [xr], [ca.gradient(rosen, xr)])
hess_rosen = ca.Function("H", [xr], [ca.hessian(rosen, xr)[0]])
print("градиент в (1,1):", grad_rosen([1, 1]))
print("гессиан  в (1,1):", hess_rosen([1, 1]))
assert np.allclose(np.array(hess_rosen([1, 1])), [[802, -400], [-400, 200]])

градиент в (1,1): [-0, 0]
гессиан  в (1,1): 
[[802, -400], 
 [-400, 200]]


**Задача 2. Солвер.** `nlpsol` получает словарь с переменными `x` и целью `f`, строит все производные сам и отдаёт задачу IPOPT — промышленному солверу ньютоновского типа. Вызов с `x0` возвращает словарь: `sol["x"]` — решение, `sol["f"]` — значение цели; `stats()` — что происходило внутри.

In [8]:
opts = {"ipopt.print_level": 0, "print_time": 0}
S = ca.nlpsol("S", "ipopt", {"x": xr, "f": rosen}, opts)
sol = S(x0=[-1.2, 1.0])
print("решение:", sol["x"], "  f* =", sol["f"], "  итераций:", S.stats()["iter_count"])
assert np.allclose(np.array(sol["x"]).ravel(), [1, 1], atol=1e-6)


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

решение: [1, 1]   f* = 3.74398e-21   итераций: 21


Пара десятков итераций — против $13\,756$ у спуска с подбором шага. Вопрос залу: *какой метод у IPOPT внутри, раз так быстро?*

**Задача 3. Спуск из лекции + производная из CasADi.** Берём цикл градиентного спуска ровно как в лекции, но производную не выписываем — берём `Function` из задачи 1. Постоянный шаг $0.001$, старт $(-1.2, 1)$: в демо вышло $32\,076$ итераций — должно сойтись.

In [9]:
def gd_const(grad, x0, alpha, tol=1e-6, maxit=100_000):
    x = np.asarray(x0, float)
    for k in range(maxit):
        g = np.array(grad(x)).ravel()      # производную считает CasADi
        if np.linalg.norm(g) <= tol:
            return x, k
        x = x - alpha * g
    return x, maxit

x_gd, k_gd = gd_const(grad_rosen, [-1.2, 1.0], alpha=0.001)
print(f"спуск с casadi-градиентом: {k_gd} итераций, x = {x_gd.round(6)}  (в демо: 32 076)")
assert abs(k_gd - 32_076) <= 50

спуск с casadi-градиентом: 32076 итераций, x = [0.999999 0.999998]  (в демо: 32 076)


**Задача 4. Квадратичная из раздела 5.** $f=\tfrac12(x_1^2+50x_2^2)$: гессиан постоянный, его собственные числа — это $1$ и $50$ с картинки множителей. Шаг $1/\lambda_{\max}$ из $(50,1)$ должен дать $1106$ итераций — число из лекции и интерактивной страницы.

In [10]:
xq = ca.SX.sym("x", 2)
fq = 0.5 * (xq[0] ** 2 + 50 * xq[1] ** 2)
Hq = np.array(ca.Function("Hq", [xq], [ca.hessian(fq, xq)[0]])(np.zeros(2)))
lam = np.linalg.eigvalsh(Hq)
print("собственные числа гессиана:", lam)

grad_q = ca.Function("gq", [xq], [ca.gradient(fq, xq)])
x_q, k_q = gd_const(grad_q, [50.0, 1.0], alpha=1 / lam[-1], tol=1e-8)
print(f"шаг 1/λ_max из (50, 1): {k_q} итераций  (в лекции: 1106)")
assert abs(k_q - 1106) <= 2

собственные числа гессиана: [ 1. 50.]
шаг 1/λ_max из (50, 1): 1106 итераций  (в лекции: 1106)


**Задача 5. Цепь.** Энергию собираем циклом по пружинам — ровно как в формуле из упражнения 12.9, — и отдаём солверу. Решение обязано совпасть с дискретной параболой $z_i=1-\frac{mg}{2D}\,i\,(N+1-i)$ и энергией $13.4417$. А один шаг Ньютона с casadi-гессианом попадает туда же: функция квадратична.

In [11]:
N, D, g_acc = 40, 70.0, 9.81
m = 4.0 / N

w = ca.SX.sym("w", 2 * N)
yc = ca.vertcat(-2, w[:N], 2)              # закреплённые концы
zc = ca.vertcat(1, w[N:], 1)
E = 0
for j in range(N + 1):                     # цикл по пружинам
    E += D / 2 * ((yc[j + 1] - yc[j]) ** 2 + (zc[j + 1] - zc[j]) ** 2)
E += m * g_acc * ca.sum1(zc[1:N + 1])      # вес грузов

S_chain = ca.nlpsol("S_chain", "ipopt", {"x": w, "f": E}, opts)
sol = S_chain(x0=np.zeros(2 * N))
v_ca = np.array(sol["x"]).ravel()

i = np.arange(1, N + 1)
y_formula = -2 + 4 * i / (N + 1)
z_formula = 1 - m * g_acc / (2 * D) * i * (N + 1 - i)
print(f"энергия: {float(sol['f']):.4f}  (руками в 12.9: 13.4417)")
print(f"max |решение - формула| = {np.abs(v_ca - np.r_[y_formula, z_formula]).max():.1e}")
assert np.allclose(v_ca, np.r_[y_formula, z_formula], atol=1e-5)

энергия: 13.4417  (руками в 12.9: 13.4417)
max |решение - формула| = 8.9e-16


In [12]:
# один шаг Ньютона с casadi-производными — из нуля сразу в минимум
grad_E = ca.Function("gE", [w], [ca.gradient(E, w)])
hess_E = ca.Function("HE", [w], [ca.hessian(E, w)[0]])
w0 = np.zeros(2 * N)
w1 = w0 - np.linalg.solve(np.array(hess_E(w0)), np.array(grad_E(w0)).ravel())
print(f"один шаг Ньютона: max |Δ с решением солвера| = {np.abs(w1 - v_ca).max():.1e}")
assert np.allclose(w1, v_ca, atol=1e-5)

один шаг Ньютона: max |Δ с решением солвера| = 1.1e-14


## Итог

- Символьная переменная — имя; выражение — дерево операций; `Function` вычисляет его, `jacobian`/`gradient`/`hessian` строят **точные** производные.
- `nlpsol` + IPOPT решают задачу за десятки итераций — внутри ньютоновские шаги с этими же производными.
- Все числа лекции воспроизвелись: $802/-400$, $32\,076$, $1106$, парабола цепи и $13.4417$.
- Чего CasADi не делает: не упрощает формулы по-человечески, не выбирает метод и не заметит плохое $\kappa$ — диагностика по графику сходимости остаётся вам.

Дома: прогнать ноутбук заново и решить 12.0–12.9 и 12.11 из конспекта; 12.10 — повторение этого семинара.